# LangGraph Exercise: Building A Basic Retriever Node
## Workshop Section 3 - Assessment Warm-Up

**Objective**: Integrate retrieval into the LangGraph agent loop to create a RAG-enabled chatbot.

**Learning Goals**:
- Implement a retrieval node that queries relevant context
- Use reranking/embedding models for semantic search
- Accumulate context over multiple conversation turns
- Add a 'deep thought' mechanism for web search fallback

## Setup and Imports

In [ ]:
import uuid
from typing import Annotated, Optional, List
from typing_extensions import TypedDict

from langgraph.checkpoint.memory import MemorySaver
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from langgraph.types import interrupt, Command
from langgraph.graph.message import add_messages
from functools import partial
from colorama import Fore, Style
from copy import deepcopy
import operator

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_nvidia import ChatNVIDIA, NVIDIARerank
from langchain_core.documents import Document
from langchain_nvidia_ai_endpoints._statics import MODEL_TABLE

from transformers import PreTrainedTokenizerFast

## LLM and Tokenizer Setup

In [ ]:
llm = ChatNVIDIA(model="meta/llama-3.1-8b-instruct", base_url="http://llm_client:9000/v1")
MODEL_TABLE[llm.model].supports_structured_output = True

llama_tokenizer = PreTrainedTokenizerFast(tokenizer_file="tokenizer.json", clean_up_tokenization_spaces=True)

def token_len(text):
    return len(llama_tokenizer.encode(text=text))

## Load Context Documents

In [ ]:
with open("simple_long_context.txt", "r") as f:
    full_context = f.read()

context_entries = full_context.split("\n\n")
context_docs = [Document(page_content=entry) for entry in context_entries if len(entry.split("\n")) > 2]

context_lens = [token_len(d.page_content) for d in context_docs]
print(f"Context Token Length: {sum(context_lens)} ({sum(context_lens)/len(context_lens):.2f} * {len(context_lens)})")
print(f"Document Token Range: [{min(context_lens)}, {max(context_lens)}]")

## Retrieval Function

In [ ]:
def retrieve_via_query(query: str, k=5):
    if not query:
        return []
    reranker = NVIDIARerank(
        model="nvidia/rerank-qa-mistral-4b",
        base_url='http://llm_client:9000/v1',
        top_n=k,
        max_batch_size=128
    )
    rets = reranker.compress_documents(context_docs, query)
    return [entry.page_content for entry in rets]

## State Definition

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]
    context: Annotated[set, (lambda x, y: x.union(y))]

## Agent Prompt

In [ ]:
agent_prompt = ChatPromptTemplate.from_messages([
    ("system",
         "You are a helpful instructor assistant for NVIDIA Deep Learning Institute (DLI). "
         "Please help to answer user questions about the course. Use the provided context."
         "Strongly rely on the context as your knowledge base. Do not refer to it as 'context'."
    ),
    ("user", "<context>\n{context}</context>"),
    ("ai", "Thank you. I will use the provided context to answer questions."),
    ("placeholder", "{messages}")
])

## User Node

In [ ]:
def user(state: State):
    update = {"messages": [("user", interrupt("[User]:"))]}
    return Command(update=update, goto="retrieval_router")

## Retrieval Router Node - EXERCISE SOLUTION

In [ ]:
def retrieval_router(state: State):
    """
    Retrieve relevant context based on the user's last message.
    Writes retrieved context to state and routes to agent.
    """
    last_message = state.get("messages")[-1].content if state.get("messages") else ""
    retrieved = retrieve_via_query(last_message, k=3)
    return Command(update={"context": set(retrieved)}, goto="agent")

## Agent Node

In [ ]:
def agent(state: State, config=None):
    if not (state.get("messages") or [""])[-1].content:
        return {}
    context_str = "\n\n".join(state.get("context", set()))
    response = (agent_prompt | llm).invoke({"context": context_str, "messages": state.get("messages")}, config=config)
    update = {"messages": [response]}
    if "stop" in state.get("messages")[-1].content:
        return update
    return Command(update=update, goto="start")

## Graph Construction

In [ ]:
builder = StateGraph(State)
builder.add_node("start", lambda state: {})
builder.add_node("user", user)
builder.add_node("retrieval_router", retrieval_router)
builder.add_node("agent", agent)
builder.add_edge(START, "start")
builder.add_edge("start", "user")
app = builder.compile(checkpointer=MemorySaver())
config = {"configurable": {"thread_id": uuid.uuid4()}}
app_stream = partial(app.stream, config=config)

## Exercise Execution

In [ ]:
from course_utils import stream_from_app

print("=" * 60)
print("RAG Chatbot - Type 'stop' to end")
print("=" * 60)
print()

for token in stream_from_app(app_stream, verbose=False, debug=False):
    print(token, end="", flush=True)

print()
print("Conversation ended!")